In [28]:
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import OrdinalEncoder
import numpy as np

# Passo 1: Preparar o conjunto de treinamento
treino = pd.read_csv('Treino_Final.csv')

# Tratar valores ausentes nas variáveis numéricas
numeric_cols = treino.select_dtypes(include=['float64', 'int64']).columns
treino[numeric_cols] = treino[numeric_cols].fillna(treino[numeric_cols].mean())

# Tratar valores ausentes nas variáveis categóricas, excluindo 'order_id'
categorical_cols = treino.select_dtypes(include=['object']).columns.drop('order_id', errors='ignore')
treino[categorical_cols] = treino[categorical_cols].fillna(treino[categorical_cols].mode().iloc[0])

# Salvar 'order_id' para referência futura (se necessário)
order_ids_train = treino['order_id']

# Remover 'order_id' das features
treino = treino.drop(columns=['order_id'], errors='ignore')

# Remover 'approval_to_carrier' e 'carrier_to_customer' das features (se desejado)
treino = treino.drop(columns=['approval_to_carrier', 'carrier_to_customer'], errors='ignore')

# Passo 2: Codificar variáveis categóricas no conjunto de treinamento
ordinal_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
treino[categorical_cols] = ordinal_encoder.fit_transform(treino[categorical_cols])

# Passo 3: Preparar os dados para treinamento
X_train = treino.drop(columns=['delivery_time (days)'])
y_train = treino['delivery_time (days)']

# Passo 4: Criar e treinar o modelo XGBoost
model = xgb.XGBRegressor(
    random_state=42,
    n_estimators=500,         # Aumente o número de estimadores
    learning_rate=0.01,       # Reduza a taxa de aprendizado
    max_depth=10,              # Profundidade das árvores
    min_child_weight=5,       # Peso mínimo por folha
    subsample=1.0,            # Subamostragem de linhas
    colsample_bytree=0.6,  
    gamma = 0.1
)
model.fit(X_train, y_train)

# Passo 5: Preparar o conjunto de teste
teste = pd.read_csv('Teste_Final.csv')

# Tratar valores ausentes nas variáveis numéricas
teste_numeric_cols = teste.select_dtypes(include=['float64', 'int64']).columns
teste[teste_numeric_cols] = teste[teste_numeric_cols].fillna(treino[numeric_cols].mean())  # Usando médias do treino

# Tratar valores ausentes nas variáveis categóricas
teste_categorical_cols = teste.select_dtypes(include=['object']).columns.drop('order_id', errors='ignore')
teste[teste_categorical_cols] = teste[teste_categorical_cols].fillna(treino[categorical_cols].mode().iloc[0])  # Usando modos do treino

# Salvar 'order_id' para uso posterior
order_ids_test = teste['order_id']

# Remover 'order_id' das features
teste = teste.drop(columns=['order_id'], errors='ignore')

# Remover 'approval_to_carrier' e 'carrier_to_customer' do conjunto de teste (caso estejam presentes)
teste = teste.drop(columns=['approval_to_carrier', 'carrier_to_customer'], errors='ignore')

# Codificar variáveis categóricas no conjunto de teste
# Garantir que usamos apenas as colunas categóricas que estão em 'categorical_cols'
teste_categorical_cols = [col for col in categorical_cols if col in teste.columns]
teste[teste_categorical_cols] = ordinal_encoder.transform(teste[teste_categorical_cols])

# Garantir que o conjunto de teste tenha as mesmas colunas que o conjunto de treinamento
missing_cols = set(X_train.columns) - set(teste.columns)
for col in missing_cols:
    teste[col] = 0  # Preencher colunas faltantes com 0 ou outro valor apropriado

# Garantir a mesma ordem de colunas
teste = teste[X_train.columns]

# Passo 6: Fazer previsões no conjunto de teste
y_pred = model.predict(teste)

# Passo 7: Criar um DataFrame com 'order_id' e a previsão
df_predictions = pd.DataFrame({
    'order_id': order_ids_test,
    'delivery_time (days)': y_pred
})

# Passo 8: Salvar as previsões em um arquivo CSV
df_predictions.to_csv('previsoes.csv', index=False)

# Opcional: Exibir as primeiras linhas das previsões
print(df_predictions.head())

                           order_id  delivery_time (days)
0  00024acbcdf0a6daa1e931b038114c75              3.667582
1  000576fe39319847cbb9d288c5617fa6              8.514414
2  0005f50442cb953dcd1d21e1fb923495              2.048056
3  00063b381e2406b52ad429470734ebd5              3.395898
4  0006ec9db01a64e59a68b2c340bf65a7              5.967933


In [41]:
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import OrdinalEncoder
import numpy as np

# Passo 1: Preparar o conjunto de treinamento
treino = pd.read_csv('Treino_Final.csv')

# Tratar valores ausentes nas variáveis numéricas
numeric_cols = treino.select_dtypes(include=['float64', 'int64']).columns
treino[numeric_cols] = treino[numeric_cols].fillna(treino[numeric_cols].mean())

# Tratar valores ausentes nas variáveis categóricas, excluindo 'order_id'
categorical_cols = treino.select_dtypes(include=['object']).columns.drop('order_id', errors='ignore')
treino[categorical_cols] = treino[categorical_cols].fillna(treino[categorical_cols].mode().iloc[0])

# Remover 'order_id' das features
treino = treino.drop(columns=['order_id'], errors='ignore')

# Passo 2: Codificar variáveis categóricas no conjunto de treinamento
ordinal_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
treino[categorical_cols] = ordinal_encoder.fit_transform(treino[categorical_cols])

# Passo 3: Separar dados para prever 'approval_to_carrier_days' e 'carrier_to_customer_days'
X_intermediate = treino.drop(columns=['approval_to_carrier_days', 'carrier_to_customer_days', 'delivery_time (days)'])
y_approval = treino['approval_to_carrier_days']
y_carrier = treino['carrier_to_customer_days']

# Passo 4: Treinar modelos para 'approval_to_carrier_days' e 'carrier_to_customer_days'

# Modelo para 'approval_to_carrier_days'
model_approval = xgb.XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_estimators=500,         # Aumente o número de estimadores
    learning_rate=0.01,       # Reduza a taxa de aprendizado
    max_depth=5,              # Profundidade das árvores
    min_child_weight=10,      # Peso mínimo por folha
    subsample=0.8,            # Subamostragem de linhas
    colsample_bytree=0.8,     # Subamostragem de colunas
    reg_alpha=0.1,            # Regularização L1
    reg_lambda=1,             # Regularização L2
    gamma=0.1                 # Reduzir a complexidade do modelo
)
model_approval.fit(X_intermediate, y_approval)

# Modelo para 'carrier_to_customer_days'
model_carrier = xgb.XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_estimators=500,         # Aumente o número de estimadores
    learning_rate=0.01,       # Reduza a taxa de aprendizado
    max_depth=5,              # Profundidade das árvores
    min_child_weight=10,      # Peso mínimo por folha
    subsample=0.8,            # Subamostragem de linhas
    colsample_bytree=0.8,     # Subamostragem de colunas
    reg_alpha=0.1,            # Regularização L1
    reg_lambda=1,             # Regularização L2
    gamma=0.1                 # Reduzir a complexidade do modelo
)
model_carrier.fit(X_intermediate, y_carrier)

# Passo 5: Preparar o conjunto de teste
teste = pd.read_csv('Teste_Final.csv')

# Tratar valores ausentes nas variáveis numéricas
teste_numeric_cols = teste.select_dtypes(include=['float64', 'int64']).columns
teste[teste_numeric_cols] = teste[teste_numeric_cols].fillna(treino[numeric_cols].mean())  # Usando médias do treino

# Tratar valores ausentes nas variáveis categóricas
teste_categorical_cols = teste.select_dtypes(include=['object']).columns.drop('order_id', errors='ignore')
teste[teste_categorical_cols] = teste[teste_categorical_cols].fillna(treino[categorical_cols].mode().iloc[0])  # Usando modos do treino

# Remover 'order_id' das features
teste = teste.drop(columns=['order_id'], errors='ignore')

# Codificar variáveis categóricas no conjunto de teste
teste_categorical_cols = [col for col in categorical_cols if col in teste.columns]
teste[teste_categorical_cols] = ordinal_encoder.transform(teste[teste_categorical_cols])

# Garantir a mesma ordem de colunas
teste_intermediate = teste[X_intermediate.columns]

# Passo 6: Prever 'approval_to_carrier_days' e 'carrier_to_customer_days' no conjunto de teste
approval_pred = model_approval.predict(teste_intermediate)
carrier_pred = model_carrier.predict(teste_intermediate)

# Passo 7: Calcular o 'delivery_time (days)' como a soma das previsões e 'purchase_to_approval_days' do conjunto de teste
teste_final = teste_intermediate.copy()
teste_final['approval_to_carrier_days'] = approval_pred
teste_final['carrier_to_customer_days'] = carrier_pred
teste_final['purchase_to_approval_days'] = teste['purchase_to_approval_days']

# Calcular 'delivery_time (days)' somando as colunas
teste_final['delivery_time (days)'] = (
    teste_final['purchase_to_approval_days'] +
    teste_final['approval_to_carrier_days'] +
    teste_final['carrier_to_customer_days']
)

# Passo 8: Criar um DataFrame com 'order_id' e 'delivery_time (days)'
df_predictions = pd.DataFrame({
    'order_id': order_ids_test,
    'delivery_time (days)': teste_final['delivery_time (days)']
})

# Passo 9: Salvar as previsões em um arquivo CSV
df_predictions.to_csv('previsoes.csv', index=False)

# Opcional: Exibir as primeiras linhas das previsões
print(df_predictions.head())


                           order_id  delivery_time (days)
0  00024acbcdf0a6daa1e931b038114c75             10.507909
1  000576fe39319847cbb9d288c5617fa6             12.520818
2  0005f50442cb953dcd1d21e1fb923495              6.666369
3  00063b381e2406b52ad429470734ebd5             10.940184
4  0006ec9db01a64e59a68b2c340bf65a7             19.042708


In [ ]:
from bayes_opt import BayesianOptimization
from sklearn.model_selection import cross_val_score
import numpy as np

# Função de otimização para 'approval_to_carrier_days'
def optimize_approval(n_estimators, learning_rate, max_depth, min_child_weight, subsample, colsample_bytree, gamma, reg_alpha, reg_lambda):
    model = xgb.XGBRegressor(
        n_estimators=int(n_estimators),  # converter para int
        learning_rate=learning_rate,
        max_depth=int(max_depth),
        min_child_weight=int(min_child_weight),
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        gamma=gamma,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=42,
        objective='reg:squarederror'
    )
    # Validação cruzada para obter a média do erro
    cv_score = cross_val_score(model, X_intermediate, y_approval, scoring='neg_mean_squared_error', cv=3)
    return np.mean(cv_score)

# Espaço de busca para 'approval_to_carrier_days'
param_bounds_approval = {
    'n_estimators': (700, 1500),  # Intervalo de 100 a 500
    'learning_rate': (0.01, 0.05),  # Intervalo de 0.01 a 0.2
    'max_depth': (3, 10),  # Intervalo de 3 a 10
    'min_child_weight': (7, 15),  # Intervalo de 1 a 10
    'subsample': (0.4, 0.7),  # Intervalo de 0.6 a 1.0
    'colsample_bytree': (0.6, 1.0),  # Intervalo de 0.6 a 1.0
    'gamma': (0, 0.5),  # Intervalo de 0 a 0.5
    'reg_alpha': (0, 0.5),  # Intervalo de 0 a 0.5
    'reg_lambda': (0.5, 2.5)  # Intervalo de 0.5 a 2.5
}

# Otimização Bayesiana para 'approval_to_carrier_days'
optimizer_approval = BayesianOptimization(
    f=optimize_approval,
    pbounds=param_bounds_approval,
    random_state=42,
    verbose=2
)
optimizer_approval.maximize(init_points=10, n_iter=40)

# Melhores parâmetros para 'approval_to_carrier_days'
best_params_approval = optimizer_approval.max['params']
best_params_approval['n_estimators'] = int(best_params_approval['n_estimators'])
best_params_approval['max_depth'] = int(best_params_approval['max_depth'])
best_params_approval['min_child_weight'] = int(best_params_approval['min_child_weight'])
print("Melhores parâmetros para 'approval_to_carrier_days':", best_params_approval)

# Função de otimização para 'carrier_to_customer_days'
def optimize_carrier(n_estimators, learning_rate, max_depth, min_child_weight, subsample, colsample_bytree, gamma):
    model = xgb.XGBRegressor(
        n_estimators=int(n_estimators),  # converter para int
        learning_rate=learning_rate,
        max_depth=int(max_depth),
        min_child_weight=int(min_child_weight),
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        gamma=gamma,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=42,
        objective='reg:squarederror'
    )
    cv_score = cross_val_score(model, X_intermediate, y_carrier, scoring='neg_mean_squared_error', cv=3)
    return np.mean(cv_score)

# Espaço de busca para 'carrier_to_customer_days'
param_bounds_carrier = {
    'n_estimators': (700, 1500),  # Intervalo de 100 a 500
    'learning_rate': (0.01, 0.05),  # Intervalo de 0.01 a 0.2
    'max_depth': (3, 10),  # Intervalo de 3 a 10
    'min_child_weight': (7, 15),  # Intervalo de 1 a 10
    'subsample': (0.4, 0.7),  # Intervalo de 0.6 a 1.0
    'colsample_bytree': (0.6, 1.0),  # Intervalo de 0.6 a 1.0
    'gamma': (0, 0.5),  # Intervalo de 0 a 0.5
    'reg_alpha': (0, 0.5),  # Intervalo de 0 a 0.5
    'reg_lambda': (0.5, 2.5)  # Intervalo de 0.5 a 2.5
}

# Otimização Bayesiana para 'carrier_to_customer_days'
optimizer_carrier = BayesianOptimization(
    f=optimize_carrier,
    pbounds=param_bounds_carrier,
    random_state=42,
    verbose=2
)
optimizer_carrier.maximize(init_points=10, n_iter=40)

# Melhores parâmetros para 'carrier_to_customer_days'
if optimizer_carrier.max is not None:
    best_params_carrier = optimizer_carrier.max['params']
    best_params_carrier['n_estimators'] = int(best_params_carrier['n_estimators'])
    best_params_carrier['max_depth'] = int(best_params_carrier['max_depth'])
    best_params_carrier['min_child_weight'] = int(best_params_carrier['min_child_weight'])
    print("Melhores parâmetros para 'carrier_to_customer_days':", best_params_carrier)
else:
    print("Otimização para 'carrier_to_customer_days' não encontrou parâmetros válidos.")


|   iter    |  target   | colsam... |   gamma   | learni... | max_depth | min_ch... | n_esti... | reg_alpha | reg_la... | subsample |
-------------------------------------------------------------------------------------------------------------------------------------
| 1         | -10.6     | 0.7498    | 0.4754    | 0.03928   | 7.191     | 8.248     | 824.8     | 0.02904   | 2.232     | 0.5803    |
| 2         | -10.74    | 0.8832    | 0.01029   | 0.0488    | 8.827     | 8.699     | 845.5     | 0.0917    | 1.108     | 0.5574    |
| 3         | -11.32    | 0.7728    | 0.1456    | 0.03447   | 3.976     | 9.337     | 993.1     | 0.228     | 2.07      | 0.4599    |
| 4         | -10.71    | 0.8057    | 0.2962    | 0.01186   | 7.253     | 8.364     | 752.0     | 0.4744    | 2.431     | 0.6425    |
| 5         | -10.74    | 0.7218    | 0.04884   | 0.03737   | 6.081     | 7.976     | 1.096e+03 | 0.01719   | 2.319     | 0.4776    |


In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import numpy as np

# Passo 1: Preparar o conjunto de treinamento
treino = pd.read_csv('Treino_Final.csv')

# Tratar valores ausentes nas variáveis numéricas
numeric_cols = treino.select_dtypes(include=['float64', 'int64']).columns
treino[numeric_cols] = treino[numeric_cols].fillna(treino[numeric_cols].mean())

# Tratar valores ausentes nas variáveis categóricas, excluindo 'order_id'
categorical_cols = treino.select_dtypes(include=['object']).columns.drop('order_id', errors='ignore')
treino[categorical_cols] = treino[categorical_cols].fillna(treino[categorical_cols].mode().iloc[0])

# Salvar 'order_id' para referência futura (se necessário)
order_ids_train = treino['order_id']

# Remover 'order_id' das features
treino = treino.drop(columns=['order_id'], errors='ignore')

# Separar variáveis categóricas para Label Encoding e OneHot Encoding
label_encode_cols = ['seller_city', 'geolocation_city_c']
onehot_encode_cols = [col for col in categorical_cols if col not in label_encode_cols]

# Passo 2: Codificar variáveis categóricas no conjunto de treinamento
# Label Encoding para 'seller_city' e 'geolocation_city_c'
label_encoders = {}
for col in label_encode_cols:
    le = LabelEncoder()
    treino[col] = le.fit_transform(treino[col])
    label_encoders[col] = le

# OneHot Encoding para as demais variáveis categóricas
onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
onehot_encoded = onehot_encoder.fit_transform(treino[onehot_encode_cols])

# Criar DataFrame com as variáveis OneHot Encoded
onehot_encoded_df = pd.DataFrame(onehot_encoded, columns=onehot_encoder.get_feature_names_out(onehot_encode_cols))

# Concatenar DataFrame original com o DataFrame OneHot Encoded
treino = pd.concat([treino.drop(columns=onehot_encode_cols), onehot_encoded_df], axis=1)

# Passo 3: Separar dados para prever 'approval_to_carrier_days' e 'carrier_to_customer_days'
# Features para predição intermediária (excluindo as colunas alvo e 'delivery_time (days)')
X_intermediate = treino.drop(columns=['approval_to_carrier_days', 'carrier_to_customer_days', 'delivery_time (days)'])
# Alvos intermediários
y_approval = treino['approval_to_carrier_days']
y_carrier = treino['carrier_to_customer_days']

# Passo 4: Treinar modelos para 'approval_to_carrier_days' e 'carrier_to_customer_days'

# Modelo para 'approval_to_carrier_days'
model_approval = xgb.XGBRegressor(
    random_state=42,
    n_estimators=500,         # Aumente o número de estimadores
    learning_rate=0.01,       # Reduza a taxa de aprendizado
    max_depth=10,             # Profundidade das árvores
    min_child_weight=5,       # Peso mínimo por folha
    subsample=1.0,            # Subamostragem de linhas
    colsample_bytree=0.6,  
    gamma=0.1
)
model_approval.fit(X_intermediate, y_approval)

# Modelo para 'carrier_to_customer_days'
model_carrier = xgb.XGBRegressor(
    random_state=42,
    n_estimators=500,
    learning_rate=0.01,
    max_depth=10,
    min_child_weight=5,
    subsample=1.0,
    colsample_bytree=0.6,  
    gamma=0.1
)
model_carrier.fit(X_intermediate, y_carrier)

# Passo 5: Adicionar as colunas de 'approval_to_carrier_days' e 'carrier_to_customer_days' ao X_train para predição final
# Usar os valores reais no conjunto de treinamento
X_train_final = X_intermediate.copy()
X_train_final['approval_to_carrier_days'] = y_approval
X_train_final['carrier_to_customer_days'] = y_carrier
y_train_final = treino['delivery_time (days)']

# Passo 6: Treinar o modelo final para prever 'delivery_time (days)'
model_final = xgb.XGBRegressor(
    random_state=42,
    n_estimators=500,
    learning_rate=0.01,
    max_depth=10,
    min_child_weight=5,
    subsample=1.0,
    colsample_bytree=0.6,  
    gamma=0.1
)
model_final.fit(X_train_final, y_train_final)

# Passo 7: Preparar o conjunto de teste
teste = pd.read_csv('Teste_Final.csv')

# Tratar valores ausentes nas variáveis numéricas
teste_numeric_cols = teste.select_dtypes(include=['float64', 'int64']).columns
teste[teste_numeric_cols] = teste[teste_numeric_cols].fillna(treino[numeric_cols].mean())  # Usando médias do treino

# Tratar valores ausentes nas variáveis categóricas
teste_categorical_cols = teste.select_dtypes(include=['object']).columns.drop('order_id', errors='ignore')
teste[teste_categorical_cols] = teste[teste_categorical_cols].fillna(treino[categorical_cols].mode().iloc[0])  # Usando modos do treino

# Salvar 'order_id' para uso posterior
order_ids_test = teste['order_id']

# Remover 'order_id' das features
teste = teste.drop(columns=['order_id'], errors='ignore')

# Label Encoding para 'seller_city' e 'geolocation_city_c'
for col in label_encode_cols:
    le = label_encoders[col]
    teste[col] = le.transform(teste[col])

# OneHot Encoding para as demais variáveis categóricas
onehot_encoded_test = onehot_encoder.transform(teste[onehot_encode_cols])

# Criar DataFrame com as variáveis OneHot Encoded
onehot_encoded_test_df = pd.DataFrame(onehot_encoded_test, columns=onehot_encoder.get_feature_names_out(onehot_encode_cols))

# Concatenar DataFrame original com o DataFrame OneHot Encoded
teste = pd.concat([teste.drop(columns=onehot_encode_cols), onehot_encoded_test_df], axis=1)

# Garantir que o conjunto de teste tenha as mesmas colunas que o X_intermediate
missing_cols = set(X_intermediate.columns) - set(teste.columns)
for col in missing_cols:
    teste[col] = 0  # Preencher colunas faltantes com 0

# Garantir a mesma ordem de colunas
teste_intermediate = teste[X_intermediate.columns]

# Passo 8: Prever 'approval_to_carrier_days' e 'carrier_to_customer_days' no conjunto de teste
approval_pred = model_approval.predict(teste_intermediate)
carrier_pred = model_carrier.predict(teste_intermediate)

# Passo 9: Adicionar as previsões ao conjunto de teste
teste_final = teste_intermediate.copy()
teste_final['approval_to_carrier_days'] = approval_pred
teste_final['carrier_to_customer_days'] = carrier_pred
teste_final['purchase_to_approval_days'] = teste['purchase_to_approval_days']

# Garantir que as colunas correspondem às do X_train_final
missing_cols_final = set(X_train_final.columns) - set(teste_final.columns)
for col in missing_cols_final:
    teste_final[col] = 0  # Preencher colunas faltantes com 0

# Ordenar as colunas
teste_final = teste_final[X_train_final.columns]

# Passo 10: Calcular 'delivery_time (days)' somando as colunas
teste_final['delivery_time (days)'] = (
    teste_final['purchase_to_approval_days'] +
    teste_final['approval_to_carrier_days'] +
    teste_final['carrier_to_customer_days']
)

# Passo 11: Criar um DataFrame com 'order_id' e 'delivery_time (days)'
df_predictions = pd.DataFrame({
    'order_id': order_ids_test,
    'delivery_time (days)': teste_final['delivery_time (days)']
})

# Passo 12: Salvar as previsões em um arquivo CSV
df_predictions.to_csv('previsoes.csv', index=False)

# Opcional: Exibir as primeiras linhas das previsões
print(df_predictions.head())